# Check how many event logs have activities with the same timestamp

In [1]:
from pm4py.objects.log.importer.xes import importer as xes_importer
from collections import defaultdict
import os
import multiprocessing
from tqdm import tqdm

def extract_traces(log_path):
    """
    Load an event log and extract all traces + unique activities.
    Also counts how many traces contain duplicated timestamps.

    Parameters:
        log_path (str): Path to the event log file

    Returns:
        tuple:
            - list: traces (with full event data + timestamp analysis)
            - set: unique activities in the log
            - int: number of traces with duplicated timestamps
    """
    
    log = xes_importer.apply(
        log_path,
        parameters={"show_progress_bar": False}
    )

    result = []
    activities = set()
    traces_with_duplicates = 0  # <- global counter

    for trace in log:
        trace_info = {
            "trace_attributes": dict(trace.attributes),
            "events": []
        }

        timestamp_counts = defaultdict(int)

        for event in trace:
            event_dict = dict(event)
            trace_info["events"].append(event_dict)

            # Collect activity name
            if "concept:name" in event_dict:
                activities.add(event_dict["concept:name"])

            # Track timestamps
            if "time:timestamp" in event_dict:
                ts = event_dict["time:timestamp"]
                timestamp_counts[ts] += 1

        # Check duplicates (only once per trace)
        duplicates = [ts for ts, count in timestamp_counts.items() if count > 1]
        if duplicates:
            traces_with_duplicates += 1  # <- increment once per trace

        result.append(trace_info)

    return result, activities, traces_with_duplicates

def _process_single_file_wrapper(file_path):
    """
    Wrapper for multiprocessing (must be top-level).
    """
    try:
        traces, _, duplicated_count = extract_traces(file_path)

        return {
            "file": os.path.basename(file_path),
            "total_traces": len(traces),
            "traces_with_duplicates": duplicated_count
        }

    except Exception as e:
        return {
            "file": os.path.basename(file_path),
            "error": str(e)
        }


def analyze_xes_folder_parallel(folder_path, max_workers=None):
    """
    Recursively scan a folder for .xes files and analyze them in parallel,
    using multiprocessing.Pool + tqdm progress bar.

    Parameters:
        folder_path (str): Root directory to search
        max_workers (int, optional): Number of processes

    Returns:
        list of dict: analysis results per file
    """

    xes_files = []

    # Collect all .xes file paths
    for root, _, files in os.walk(folder_path):
        for file in files:
            if file.lower().endswith(".xes"):
                xes_files.append(os.path.join(root, file))

    num_cores = max_workers or (os.cpu_count() or 1)

    # Parallel processing with progress bar
    with multiprocessing.Pool(num_cores) as pool:
        results = list(
            tqdm(
                pool.imap(_process_single_file_wrapper, xes_files),
                total=len(xes_files),
                desc="Processing XES files",
                unit="file"
            )
        )

    return results

def summarize_duplicates_text(results):
    """
    Print a textual summary of duplicated timestamps per file and overall.

    Parameters:
        results (list of dict): Output from analyze_xes_folder_parallel
    """

    # Filter out files with errors
    valid_results = [r for r in results if "error" not in r]

    if not valid_results:
        print("No valid results to summarize.")
        return

    files_with_duplicates_count = 0

    for r in valid_results:
        total_traces = r["total_traces"]
        duplicated_traces = r["traces_with_duplicates"]
        pct = (duplicated_traces / total_traces * 100) if total_traces > 0 else 0

        if duplicated_traces > 0:
            files_with_duplicates_count += 1

        print(f"Event Log: {r['file']}, total traces: {total_traces}, "
              f"Traces with shared ts: {duplicated_traces}, percentage: {pct:.1f}%")

    total_files = len(valid_results)
    print(f"\n{files_with_duplicates_count} out of {total_files} event logs have trace with duplicated timestamps.")

In [2]:
path = "all_event_logs"
results = analyze_xes_folder_parallel(path,22)

Processing XES files:   0%|                            | 0/21 [00:00<?, ?file/s]/opt/conda/lib/python3.10/site-packages/pm4py/util/dt_parsing/parser.py:82: UserWarning: ISO8601 strings are not fully supported with strpfromiso for Python versions below 3.11
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/pm4py/util/dt_parsing/parser.py:82: UserWarning: ISO8601 strings are not fully supported with strpfromiso for Python versions below 3.11
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/pm4py/util/dt_parsing/parser.py:82: UserWarning: ISO8601 strings are not fully supported with strpfromiso for Python versions below 3.11
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/pm4py/util/dt_parsing/parser.py:82: UserWarning: ISO8601 strings are not fully supported with strpfromiso for Python versions below 3.11
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/pm4py/util/dt_parsing/parser.py:82: UserWarning: ISO8601 strings are not fully supported with strpfromis

In [3]:
summarize_duplicates_text(results)

Event Log: BPIC15_3.xes, total traces: 1409, Traces with shared ts: 1381, percentage: 98.0%
Event Log: BPIC15_5.xes, total traces: 1156, Traces with shared ts: 1153, percentage: 99.7%
Event Log: BPIC15_1.xes, total traces: 1199, Traces with shared ts: 1170, percentage: 97.6%
Event Log: BPIC15_2.xes, total traces: 832, Traces with shared ts: 827, percentage: 99.4%
Event Log: BPI_Challenge_2019.xes, total traces: 251734, Traces with shared ts: 19365, percentage: 7.7%
Event Log: BPIC15_4.xes, total traces: 1053, Traces with shared ts: 1047, percentage: 99.4%
Event Log: InternationalDeclarations.xes, total traces: 6449, Traces with shared ts: 374, percentage: 5.8%
Event Log: BPI_Challenge_2013_incidents.xes, total traces: 7554, Traces with shared ts: 0, percentage: 0.0%
Event Log: DomesticDeclarations.xes, total traces: 10500, Traces with shared ts: 10, percentage: 0.1%
Event Log: Road_Traffic_Fine_Management_Process.xes, total traces: 150370, Traces with shared ts: 9166, percentage: 6.1%
